In [23]:
# general import: 
import sys
import time
import os
import gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [24]:
# quantum import:
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix
from qiskit.visualization import plot_state_city, plot_state_hinton, plot_state_qsphere

In [25]:
# ML import:
import tensorflow as tf
import keras
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
import sklearn
from sklearn.model_selection import train_test_split

In [26]:
# custom helper and libraries:
sys.path.append('../src')
from povm_sampling import *
from statesprep import *
from vae import *
from plots import *
from utils import *
from mle import *

# Intervalli di Confidenza su $F_c$

Userò 100 dataset indipendenti da 500 sample ciascuno

In [ ]:
def genera_dataset(p_exact, seed_data, n_samples=500):
    samples = sample_povm(p_exact, n_samples, seed=seed_data)
    return samples

In [ ]:
def fit_mle(p_exact, samples, N):
    nll = make_nll(samples, N)

    dim = 2**N
    init_rho = np.eye(dim, dtype=complex) / dim
    p0 = rho_to_params(init_rho, N)

    m = Minuit(nll, *p0)
    m.errordef = Minuit.LIKELIHOOD
    m.print_level = 0
    m.migrad()
    if not m.valid:
        m.migrad()                                

    rho_mle = params_to_rho(np.array(m.values), N)

    p_mle = povm_probability(rho_mle, N)           # distribuzione ricostruita
    fc = classical_fidelity(p_exact, p_mle)

    return float(fc)

In [ ]:
def fit_vae(p_exact, samples, N, seed_train, n_gen=50_000):
    keras.utils.set_random_seed(seed_train)        # <-- fissa init/shuffle

    onehot_samples = samples_to_onehot(samples, N)

    X_train, X_test = train_test_split(onehot_samples, test_size=0.2,
                                       random_state=42)

    LATENT_DIM = 32
    HIDDEN_DIM = 128
    WARMUP_EPOCHS = 50
    TOTAL_EPOCHS = 500
    BATCH_SIZE = 100
    LEARNING_RATE = 1e-3
    BETA_MAX = 0.85

    vae = VAE(n_qubits=N, latent_dim=LATENT_DIM, hidden=HIDDEN_DIM)
    vae.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE))
    vae.build(input_shape=(None, 4*N))

    callbacks = [
        KLWarmup(beta_max=BETA_MAX, warmup_epochs=WARMUP_EPOCHS),
        keras.callbacks.EarlyStopping(
            monitor='val_reconstruction_loss', mode='min', patience=50,
            restore_best_weights=True, start_from_epoch=WARMUP_EPOCHS),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_reconstruction_loss', mode='min',
            factor=0.5, patience=20, min_lr=1e-6),
    ]
    vae.fit(X_train, X_train, epochs=TOTAL_EPOCHS, batch_size=BATCH_SIZE,
            validation_data=(X_test, X_test), callbacks=callbacks, verbose=0)

    p_vae = vae.predict_dist(n_samples=n_gen, batch_size=50_000)
    fc = classical_fidelity(p_exact, p_vae)

    # libera grafo TF + RAM: in un kernel longevo clear_session() da solo non basta,
    # serve anche eliminare il modello e forzare il garbage collector.
    del vae
    keras.backend.clear_session()
    gc.collect()
    return float(fc)

In [ ]:
N = 4
qc = create_ghz_state(N)
rho_true = DensityMatrix(qc)
n_samples = 200
p_exact = povm_probability(rho_true.data, N)

In [ ]:
B = 50                          # numero di dataset indipendenti (stats.md: 50-100)
N_GEN = 50_000                   # campioni VAE per stimare p_vae (>> n_samples, vedi stats.md)
SEED_TRAIN_FISSO = 12345  

rows = []
for b in range(B):
    samples = genera_dataset(p_exact=p_exact, seed_data=10_000 + b, n_samples=n_samples)
    fc_m = fit_mle(p_exact, samples, N)                                                  
    fc_v = fit_vae(p_exact, samples, N, seed_train=SEED_TRAIN_FISSO, n_gen=N_GEN)     

    # stesso dataset b per entrambi => confronto PAIRED
    rows.append(dict(method='MLE', b=b, N=n_samples, F_c=fc_m))
    rows.append(dict(method='VAE', b=b, N=n_samples, F_c=fc_v))

    if b % 1 == 0:
        print(f"{b+1}/{B}  F_c: MLE={fc_m:.4f}  VAE={fc_v:.4f}")

# UN solo CSV, colonna 'b' = indice replica
df = pd.DataFrame(rows)
df.to_csv(f'risultati_shot_N{n_samples}_B{B}.csv', index=False)
print("\nSalvato. Righe:", len(df))

1/50  F_c: MLE=0.9692  VAE=0.9437
2/50  F_c: MLE=0.9746  VAE=0.9490


KeyboardInterrupt: 

In [ ]:
for met in ['MLE', 'VAE']:
    v = df.query("method == @met")['F_c'].values
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{met}:  F_c = {v.mean():.4f}   CI95 = [{lo:.4f}, {hi:.4f}]")
 
# Confronto paired: differenza VAE - MLE sugli stessi dataset
piv = df.pivot_table(index='b', columns='method', values='F_c')
delta = (piv['VAE'] - piv['MLE']).values
print(f"\nDelta F_c (VAE-MLE): media = {delta.mean():.4f}")
print(f"frazione dataset con VAE meglio di MLE: {(delta > 0).mean():.2%}")